Below is a **simple Agent example** using **LangChain + Azure OpenAI**. This is the type of example you can explain comfortably in an interview.

---

# Architecture

```text
                User Question
                      │
                      ▼
            Azure OpenAI (GPT-4o)
                      │
              Agent Reasoning
                      │
          ┌───────────┴───────────┐
          ▼                       ▼
      Calculator Tool        Search Tool
          │                       │
          └───────────┬───────────┘
                      ▼
            Azure OpenAI (GPT-4o)
                      │
                      ▼
                 Final Answer
```

---

# Install

```bash
pip install langchain
pip install langchain-openai
pip install langchain-community
```

---

# Step 1: Configure Azure OpenAI

```python
import os
from langchain_openai import AzureChatOpenAI

os.environ["AZURE_OPENAI_API_KEY"] = "YOUR_API_KEY"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-openai.openai.azure.com/"
os.environ["OPENAI_API_VERSION"] = "2024-02-15-preview"

llm = AzureChatOpenAI(
    azure_deployment="gpt-4o",
    api_version="2024-02-15-preview",
    temperature=0
)
```

---

# Step 2: Create Tools

```python
from langchain.tools import tool

@tool
def calculator(expression: str):
    """Evaluate a mathematical expression."""
    return eval(expression)

@tool
def company_info(company: str):
    """Returns company information."""

    data = {
        "Altimetrik": "Altimetrik is a digital engineering company.",
        "Microsoft": "Microsoft develops Azure and OpenAI services."
    }

    return data.get(company, "Company not found.")
```

---

# Step 3: Create Agent

```python
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
```

---

# Step 4: Prompt

```python
prompt = ChatPromptTemplate.from_messages(
[
("system","You are a helpful AI Assistant."),
("human","{input}"),
("placeholder","{agent_scratchpad}")
]
)
```

---

# Step 5: Build Agent

```python
tools = [calculator, company_info]

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)
```

---

# Step 6: Invoke

```python
response = agent_executor.invoke(
{
"input":"What is 250 * 40?"
})

print(response["output"])
```

---

Another example:

```python
response = agent_executor.invoke(
{
"input":"Tell me about Altimetrik"
})

print(response["output"])
```

---

# What Happens Internally?

```
User:
"What is 250 * 40?"

        │

        ▼

Azure OpenAI

        │

Reasoning:
"I need Calculator Tool."

        │

        ▼

Calculator Tool

250 * 40

↓

10000

        │

        ▼

Azure OpenAI

↓

Final Response

"The answer is 10,000."
```

---

# Interview Explanation (1 Minute)

> The user sends a query to the LangChain agent. Azure OpenAI first reasons about the request and decides whether it needs a tool. If the query requires a calculation, it calls the calculator tool. If it needs company information, it calls the company information tool. The tool output is returned to the LLM, which then generates the final natural-language response. This demonstrates tool calling and agent orchestration.

---

# If They Ask: "Why use an Agent instead of calling the LLM directly?"

**Answer:**

A normal LLM only generates text.

An **Agent** can:

- Decide what action to take.
- Select the appropriate tool.
- Call external APIs.
- Query databases.
- Perform calculations.
- Execute multiple steps before producing the final answer.

---

# Difference Between RAG and Agent

| RAG | Agent |
|------|--------|
| Retrieves documents | Chooses and executes tools |
| Answers using retrieved context | Can take actions and make decisions |
| Uses Retriever + LLM | Uses LLM + Tools + Reasoning |
| Knowledge-focused | Task- and workflow-focused |
| Example: PDF chatbot | Example: Travel booking, claims processing, SQL agent |

---

### Interview Tip

If the interviewer asks how this changes for **Azure**, the answer is simple:

> "Only the model provider changes. The agent logic, tools, prompts, and LangChain orchestration remain the same. Instead of using Bedrock's `ChatBedrock`, I use `AzureChatOpenAI`, authenticated with Azure OpenAI (preferably via Managed Identity in production)."